# 01 · Tarefa, base de conhecimento e gabarito
**GS 2026.2 · CitizenTech · Generative AI for Engineering — LLM local para a Defesa Civil**

**Tarefa.** Um assistente local (Qwen2.5-3B-Instruct, sem enviar dados a APIs externas) que responde, em português, perguntas de operadores e brigadistas sobre protocolos de incêndio (florestal e em edificações) **usando somente** uma base de 9 PDFs oficiais. Cada resposta é um JSON:

```json
{"status": "ok" | "nao_consta", "resposta": "texto", "citacoes": [{"chunk_id": "DOC005-p039-c041"}]}
```

- `ok` exige citação de trecho realmente fornecido; o sistema enriquece a citação com documento, página e trecho original (explicabilidade).
- `nao_consta` é a resposta correta quando a base não contém a informação (em vez de inventar).

**Configurações comparadas:** A = modelo base sem contexto · B = RAG simples (busca + limiar) · C = RAG + LoRA.

Este notebook: (1) ingere os PDFs em trechos com fonte, (2) tabela da base, (3) valida o gabarito contra o texto real dos PDFs.

In [1]:
# --- Ambiente: funciona no Colab (via VSCode ou navegador) e localmente ---------------
import os, sys, subprocess
from pathlib import Path

REPO_URL = ""   # opcional: URL do repositório GitHub, se o projeto ainda não estiver na máquina do Colab

try:
    os.getcwd()
except FileNotFoundError:      # a pasta atual do kernel foi apagada/recriada: volta para um lugar válido
    os.chdir("/content" if Path("/content").exists() else str(Path.home()))

def achar_raiz():
    cwd = Path.cwd()
    cands = [cwd, *cwd.parents, Path("/content/GS2-GENIA"), Path("/content/drive/MyDrive/GS2-GENIA"),
             Path("/content/drive/MyDrive/GEN-IA/GS2-GENIA"), Path("/content/drive/MyDrive/GEN-IA")]
    for c in cands:
        if (c / "src" / "config.py").exists():
            return c
    for base in (Path("/content"), Path("/content/drive/MyDrive")):   # busca 1-2 níveis abaixo
        if base.exists():
            for f in list(base.glob("*/src/config.py")) + list(base.glob("*/*/src/config.py")):
                return f.parent.parent
    return None

ROOT = achar_raiz()
if ROOT is None and REPO_URL:
    subprocess.run(["git", "clone", REPO_URL, "/content/GS2-GENIA"], check=True)
    ROOT = Path("/content/GS2-GENIA")
assert ROOT is not None, "Não achei src/config.py: abra o notebook dentro da pasta GS2-GENIA ou preencha REPO_URL."
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

ROOT = /content/drive/MyDrive/GS2-GENIA


In [2]:
%pip install -q -r requirements.txt

In [3]:
import json, shutil, pandas as pd
from src import config
from src.ingest import build_chunks, save_chunks, load_docs
from src.evaluate import load_testset, validate_testset, check_testset_against_chunks

docs = load_docs()
chunks, report = build_chunks(docs)
save_chunks(chunks)
print(len(chunks), "trechos salvos em", config.CHUNKS_PATH)
pd.DataFrame(report)

1312 trechos salvos em /content/drive/MyDrive/GS2-GENIA/data/chunks.jsonl


,id,arquivo,paginas,paginas_sem_texto,chunks,possivel_scan
0,DOC001,20250428_roteiro_metodologico.pdf,46,6,113,False
1,DOC002,GUIA-PRATICO-PMIF.pdf,37,4,95,False
2,DOC003,cartilhaPNMIF.pdf,36,4,41,False
3,DOC004,E-possivel-controlar-fogo-vs-site.pdf,4,0,8,False
4,DOC005,2010_ICMBIO_ApostilaBrigadistaPrevencaoIncendi...,90,11,175,False
5,DOC006,20250311_Manual_22661453_Manual_SCI_100_Introd...,85,2,167,False
6,DOC007,20250311_Manual_22661456_Manual_SCI_200_Basico...,143,12,307,False
7,DOC008,20250311_Manual_22661458_Manual_do_Curso_Inter...,181,5,363,False
8,DOC009,instrucao_tecnica_16_2015_plano_de_emergencia_...,13,0,43,False


## Base de conhecimento (fonte, formato, licença, nº de trechos)
`paginas_sem_texto` alto indicaria PDF escaneado (precisaria de OCR). A licença abaixo é o que cada documento declara; quando o PDF não declara, está escrito assim — **não presumimos licença aberta**.

In [4]:
LICENCAS = {
    "DOC001": "Publicação institucional (IBAMA/Prevfogo); licença não declarada no PDF - uso acadêmico, citando a fonte",
    "DOC002": "Publicação institucional (ISPN/IBAMA); licença não declarada no PDF - uso acadêmico, citando a fonte",
    "DOC003": "Publicação institucional (IBAMA); licença não declarada no PDF - uso acadêmico, citando a fonte",
    "DOC004": "Material informativo (ISPN); licença não declarada no PDF - uso acadêmico, citando a fonte",
    "DOC005": "Publicação institucional (ICMBio/MMA); licença não declarada no PDF - uso acadêmico, citando a fonte",
    "DOC006": "Manual SCI: reprodução permitida sem alteração, citando a fonte e sem fins comerciais (nota do manual)",
    "DOC007": "Manual SCI: reprodução permitida sem alteração, citando a fonte e sem fins comerciais (nota do manual)",
    "DOC008": "Manual SCI: reprodução permitida sem alteração, citando a fonte e sem fins comerciais (nota do manual)",
    "DOC009": "Norma técnica do CBPMESP (IT 16); licença não declarada - verificar vigência (cabeçalho do PDF traz 'IT 16/2011')",
}
por_doc = pd.Series([c["doc_id"] for c in chunks]).value_counts().to_dict()
rows = []
for d in docs:
    r = next((x for x in report if x["id"] == d["id"]), {})
    rows.append({"doc_id": d["id"], "titulo": d["titulo"], "instituicao": d["instituicao"],
                 "formato": "PDF", "status": d.get("status", ""), "licenca": LICENCAS.get(d["id"], "?"),
                 "paginas": r.get("paginas"), "chunks": por_doc.get(d["id"], 0)})
fontes = pd.DataFrame(rows)
display(fontes)
print("Total de trechos:", int(fontes.chunks.sum()))

(ROOT / "base").mkdir(exist_ok=True)
shutil.copy(config.CHUNKS_PATH, ROOT / "base" / "chunks.jsonl")
fontes.to_csv(ROOT / "base" / "fontes.csv", index=False)

,doc_id,titulo,instituicao,formato,status,licenca,paginas,chunks
0,DOC001,Roteiro Metodológico para Elaboração de Planos...,IBAMA / Prevfogo,PDF,referencia_tecnica,Publicação institucional (IBAMA/Prevfogo); lic...,46,113
1,DOC002,Guia Prático de Manejo Integrado do Fogo,ISPN / IBAMA,PDF,referencia_tecnica,Publicação institucional (ISPN/IBAMA); licença...,37,95
2,DOC003,Manejo Integrado do Fogo,IBAMA,PDF,referencia_institucional,Publicação institucional (IBAMA); licença não ...,36,41
3,DOC004,É possível controlar o fogo,ISPN,PDF,historico,Material informativo (ISPN); licença não decla...,4,8
4,DOC005,Apostila Brigadista de Prevenção e Combate,ICMBio / MMA,PDF,referencia_tecnica,Publicação institucional (ICMBio/MMA); licença...,90,175
5,DOC006,Introdução ao Sistema de Comando de Incidentes,IBAMA / ICMBio / USFS,PDF,referencia_tecnica,Manual SCI: reprodução permitida sem alteração...,85,167
6,DOC007,Sistema de Comando de Incidentes - Resposta In...,IBAMA / USFS,PDF,referencia_tecnica,Manual SCI: reprodução permitida sem alteração...,143,307
7,DOC008,SCI para Incidentes em Expansão,IBAMA / ICMBio / USFS,PDF,referencia_tecnica,Manual SCI: reprodução permitida sem alteração...,181,363
8,DOC009,Instrução Técnica 16 - Plano de Emergência,Corpo de Bombeiros de São Paulo,PDF,verificar_vigencia,Norma técnica do CBPMESP (IT 16); licença não ...,13,43


Total de trechos: 1312


## Gabarito (40 perguntas: 28 com resposta + 12 "não consta")
Cada pergunta respondível traz documento e **páginas físicas** esperados, resposta de referência e `fatos_chave`: grupos de termos curtos (alternativas aceitas dentro do grupo, todos os grupos obrigatórios), comparados em texto normalizado. As 12 perguntas "não consta" são de três tipos: fora do domínio (6), temas próximos do domínio com número/regra específico que a base **não** traz (6; ex.: vento máximo em km/h para queima prescrita) — a ausência foi conferida por busca textual nos PDFs.

Divisão: 20 `dev` (calibra limiar) e 20 `teste` (resultado final), cada uma com respondíveis e "não consta".

In [5]:
items = load_testset()
print("Problemas de estrutura:", validate_testset(items, docs, chunks) or "nenhum")
probs = check_testset_against_chunks(items, chunks)
print("Gabarito x texto real dos PDFs:", probs or "OK - todos os fatos_chave aparecem nas páginas esperadas")
t = pd.DataFrame(items)
display(pd.crosstab(t.split, t.tipo, margins=True))
shutil.copy(config.TESTSET_PATH, ROOT / "avaliacao" / "testset.jsonl") if (ROOT / "avaliacao").exists() else None

Problemas de estrutura: nenhum
Gabarito x texto real dos PDFs: OK - todos os fatos_chave aparecem nas páginas esperadas


tipo,definicao,fato,nao_consta,procedimento,All
split,,,,,
dev,6,6,6,2,20
teste,3,6,6,5,20
All,9,12,12,7,40


PosixPath('/content/drive/MyDrive/GS2-GENIA/avaliacao/testset.jsonl')

**Se a célula acima listar problemas**, o gabarito (página ou fato-chave) não bate com o texto extraído: corrija `data/testset.jsonl` antes de seguir — nenhuma métrica dos notebooks seguintes é confiável com gabarito errado. (Páginas são as do PDF, 1-based; o número impresso no rodapé de alguns PDFs é diferente.)